# KNN avec PCA et LDA

Ce notebook implémente deux pipelines de classification KNN, l'un avec une réduction de dimension par PCA et l'autre par LDA. On compare les deux approches sur le même jeu de données.

## Imports

On importe toutes les bibliothèques nécessaires pour les deux pipelines.

In [ ]:
# pyright: ignore
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from import_ipynb import NotebookFinder  # type: ignore
import importlib
import joblib
import os
import pandas as pd
from huggingface_hub import hf_hub_download
import shutil
from import_ipynb import NotebookFinder  # type: ignore
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score
import importlib
import os
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np
import json
import numpy as np
from tqdm import tqdm

## Chargement des modules utilitaires

On définit les chemins vers les dossiers du projet et on charge les modules `metrics` et `prints` qui contiennent les fonctions d'évaluation et d'affichage.

In [ ]:
# Retrouver les dossiers
root = r"C:\Travaux\Epitech\Zoidberg2.0"

hyperparameter_dir = os.path.join(root, "pneumonia_knn", "document", "results", "model", "hyperparameter")
utility_dir = os.path.join(root, "pneumonia_knn", "utility")
pipeline_dir = os.path.join(root, "pneumonia_knn", "pipeline")
dataset_dir = os.path.join(root, "pneumonia_knn", "document", "result","model", "dataset")
result_dir = os.path.join(root, "pneumonia_knn","document" ,"result")

# charger les fichiers
# --- on_the_fly_augmentation.ipynb
spec_on_the_fly_augmentation = NotebookFinder().find_spec("on_the_fly_augmentation", [root])
on_the_fly_augmentation = importlib.util.module_from_spec(spec_on_the_fly_augmentation)
spec_on_the_fly_augmentation.loader.exec_module(on_the_fly_augmentation)

# --- pneumonia_knn\utility\dataset.ipynb
spec_utility_dataset = NotebookFinder().find_spec("dataset", [utility_dir])
utility_dataset = importlib.util.module_from_spec(spec_utility_dataset)
spec_utility_dataset.loader.exec_module(utility_dataset)

# --- pneumonia_knn\notebooks\process\knn_pca.ipynb
spec_knn_pca = NotebookFinder().find_spec("knn_pca", [pipeline_dir])
knn_pca = importlib.util.module_from_spec(spec_knn_pca)

# --- pneumonia_knn\notebooks\process\knn_lda.ipynb
spec_knn_lda = NotebookFinder().find_spec("knn_lda", [pipeline_dir])
knn_lda = importlib.util.module_from_spec(spec_knn_lda)

# --- pneumonia_knn\notebooks\utility
spec_print = NotebookFinder().find_spec("prints", [utility_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

# Charger le module metrics
spec_metrics = NotebookFinder().find_spec("metrics", [utility_dir])
metrics = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics)

# Charger le module prints
spec_print = NotebookFinder().find_spec("prints", [utility_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

# Test

In [ ]:
def apply_dataset_to_array(dataset, name):
    X = []
    y = []
    for example in tqdm(dataset, desc=f"Processing {name} dataset"):
        img_tensor = example['image']
        img_flattened = np.array(img_tensor).flatten()
        X.append(img_flattened)
        y.append(example['label'])
    return np.array(X), np.array(y)

# Test

In [ ]:
def cross_validation_png(grid, implementation_name):
    results = pd.DataFrame(grid.cv_results_)  # ← ajoute cette ligne
    # Sélectionner les colonnes clés
    table_df = results[[
        'params',
        'mean_test_accuracy',
        'mean_test_precision',
        'mean_test_recall',
        'mean_test_f1',
        'mean_test_roc_auc',
        'rank_test_accuracy'
    ]].copy()
    table_df.insert(0, 'Index CV', [f'#{i+1}' for i in range(len(table_df))])  # ← ici

    # Afficher seulement l'index dans la colonne params
    table_df['params'] = [str(i+1) for i in range(len(table_df))]
    
    # Supprimer la colonne params (redondante avec Index CV)
    table_df = table_df.drop(columns=['params'])

    # Ajuster la taille en fonction du nombre de lignes et colonnes
    num_cols = len(table_df.columns)
    num_rows = len(table_df)
    fig_width = 2 + num_cols * 1.8  # largeur adaptée au nombre de colonnes
    fig_height = 1.5 + num_rows * 0.45  # hauteur adaptée au nombre de lignes
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc='center',
        loc='center',
        colColours=['#f7f7f7'] + ['#ffffff'] * (num_cols - 1)
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)

    # Trouver l'indice de la ligne avec rank_test_accuracy = 1
    rank_col_idx = table_df.columns.get_loc('rank_test_accuracy')
    best_rank_row = None
    for i in range(num_rows):
        if table_df.iloc[i, rank_col_idx] == 1:
            best_rank_row = i + 1  # +1 car la ligne 0 est l'en-tête
            break

    # Styliser l'en-tête
    for j in range(num_cols):
        table[(0, j)].set_facecolor('#40466e')
        table[(0, j)].set_text_props(weight='bold', color='white')

    # Alternance de couleur sur les lignes + surbrillance du meilleur rang
    for i in range(1, num_rows + 1):
        for j in range(num_cols):
            if i == best_rank_row:
                # Surbrillance pour le meilleur rang
                table[(i, j)].set_facecolor('#FFD700')  # Doré
                table[(i, j)].set_text_props(weight='bold')
            elif i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')

    plt.title(implementation_name + " Cross-Validation Results", fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    output_path = f'{result_dir}/fine_tuning/cross_validation_{implementation_name}.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.show()

# Test

In [ ]:
def cross_validation_json(grid, implementation_name):
    # Préparer les données à sauvegarder
    results_data = {
        'implementation': implementation_name,
        'best_params': grid.best_params_,
        'best_score': float(grid.best_score_),
        'best_index': int(grid.best_index_),
        'cv_results': {}
    }
    
    # Convertir les colonnes numpy en listes pour la sérialisation JSON
    for key, value in grid.cv_results_.items():
        if hasattr(value, 'tolist'):
            results_data['cv_results'][key] = value.tolist()
        else:
            results_data['cv_results'][key] = value
    
    # Sauvegarder en JSON
    output_path = f'{result_dir}/fine_tuning/cross_validation_{implementation_name}.json'
    with open(output_path, 'w') as f:
        json.dump(results_data, f, indent=4)

# Test

In [ ]:
hf_token = os.getenv("KEY_HUGGING_FACE")
local_paths = [
    os.path.join(dataset_dir, "dataset_train.pkl"),
    os.path.join(dataset_dir, "dataset_val.pkl"),
    os.path.join(dataset_dir, "dataset_test.pkl")
   ]
for local_path in local_paths:
    filename = os.path.basename(local_path)
    if not os.path.exists(local_path):
        print("Que souhaitez-vous faire ?")
        print("1. Utiliser le dataset déjà processé")
        print("2. Lancer le preprocessing du dataset")
        choice = input("Entrez 1 ou 2 : ")
        if choice == "1":
            print("Vous avez choisi de télécharger le dataset déjà processé deuis un repo HuggingFace.")
            try:
                print("Téléchargement en cours...")
                cached_path = hf_hub_download(
                    repo_id="Aidavef/chest-xray-pneumonia",
                    filename=f"KNN/models/v1/{filename}",
                    repo_type="model",
                    token=hf_token,
                    local_dir=dataset_dir,
                    local_files_only=False,
                    cache_dir=None  # Désactiver le cache
                )
                # Déplacer le fichier directement dans dataset_dir
                dest_path = os.path.join(dataset_dir, filename)
                shutil.move(cached_path, dest_path)

                # Nettoyer le dossier KNN créé automatiquement
                shutil.rmtree(os.path.join(dataset_dir, "KNN"), ignore_errors=True)
            except Exception as e:
                raise e
        elif choice == "2":
            print("Vous avez choisi de lancer le processing du dataset.")
            if filename == "dataset_train.pkl":
                joblib.dump((utility_dataset.apply_dataset_to_array(on_the_fly_augmentation.train_data, "train")), f'{dataset_dir}/dataset_train.pkl')
            elif filename == "dataset_test.pkl":
                joblib.dump((utility_dataset.apply_dataset_to_array(on_the_fly_augmentation.test_data, "test")), f'{dataset_dir}/dataset_test.pkl')
            elif filename == "dataset_val.pkl":
                joblib.dump((utility_dataset.apply_dataset_to_array(on_the_fly_augmentation.val_data, "val")), f'{dataset_dir}/dataset_val.pkl')

# charger les datasets
X_train, y_train = joblib.load(f'{dataset_dir}/dataset_train.pkl')
X_val, y_val = joblib.load(f'{dataset_dir}/dataset_val.pkl')
X_test, y_test = joblib.load(f'{dataset_dir}/dataset_test.pkl')


---
# Pipeline PCA + KNN

## Création du pipeline

On crée un pipeline avec deux étapes : la réduction de dimension avec PCA, puis la classification avec KNN.

Les hyperparamètres testés sont :
- `pca__n_components` : 0.95 (on garde 95% de la variance)
- `knn__n_neighbors` : 3, 5, 9
- `knn__metric` : euclidean, manhattan
- `knn__weights` : distance
- `knn__algorithm` : ball_tree

In [ ]:
# Pipeline PCA + KNN
pipeline_pca = Pipeline([
    ('pca', PCA()),
    ('knn', KNeighborsClassifier())
])

# Hyperparamètres à tester
param_grid_pca = {
    'pca__n_components': [0.95],
    'knn__n_neighbors': [3, 5, 9],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__weights': ['distance'],
    'knn__algorithm': ['ball_tree']
}

## GridSearchCV — PCA + KNN

On lance une recherche par grille pour trouver les meilleurs hyperparamètres. Si le fichier `.pkl` existe déjà, on le charge directement pour éviter de tout relancer.

In [ ]:
file_name_pca = "knn_pca_grid_search.pkl"

if not os.path.exists(f'{hyperparameter_dir}/{file_name_pca}'):
    prints.bold("Process : Cross Validation - PCA + KNN")
    grid_pca = GridSearchCV(
        estimator=pipeline_pca,
        param_grid=param_grid_pca,
        cv=2,
        scoring=metrics.get_scoring(),
        refit='recall',
        n_jobs=1,
        verbose=2
    )
    grid_pca.fit(X_train, y_train)  # type: ignore
    joblib.dump(grid_pca, f'{hyperparameter_dir}/{file_name_pca}')

grid_pca = joblib.load(f'{hyperparameter_dir}/{file_name_pca}')

## Transformation PCA

On applique la transformation PCA du meilleur modèle trouvé sur les données d'entraînement et de test.

In [ ]:
X_train_pca = grid_pca.best_estimator_.named_steps['pca'].transform(X_train)  # type: ignore
X_test_pca  = grid_pca.best_estimator_.named_steps['pca'].transform(X_test)   # type: ignore

## Prédictions — PCA + KNN

On génère les prédictions sur les données de test avec le meilleur pipeline PCA + KNN.

In [ ]:
knn_pca   = grid_pca.best_estimator_
y_pred_pca = knn_pca.predict(X_test)  # type: ignore

## Résultats — PCA + KNN

On affiche les résultats de la validation croisée sous forme de tableau et on les sauvegarde en JSON.

In [ ]:
cross_validation_png(grid_pca, "PCA")
cross_validation_json(grid_pca, "PCA")

---
# Pipeline LDA + KNN

## Création des pipelines

On crée deux variantes :
- **LDA seul** : StandardScaler + LDA + KNN (les données brutes nécessitent une normalisation avant LDA)
- **PCA + LDA** : LDA + KNN (les données déjà réduites par PCA n'ont plus besoin du scaler)

Les hyperparamètres testés sont :
- `knn__n_neighbors` : 3, 5, 9
- `knn__metric` : euclidean, manhattan
- `knn__weights` : distance
- `knn__algorithm` : brute

In [ ]:
# Pipeline LDA seul (avec scaler car données brutes)
pipeline_lda = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LDA(n_components=2)),
    ('knn', KNeighborsClassifier())
])

# Pipeline PCA + LDA (pas de scaler, X_train_pca déjà normalisé)
pipeline_pca_lda = Pipeline([
    ('lda', LDA(n_components=2)),
    ('knn', KNeighborsClassifier())
])

# Hyperparamètres à tester
param_grid_lda = {
    'knn__n_neighbors': [3, 5, 9],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__weights': ['distance'],
    'knn__algorithm': ['brute']
}

## GridSearchCV — LDA + KNN

Même logique que pour PCA : on lance la recherche par grille ou on charge le fichier déjà sauvegardé. La variable `implementation` permet de choisir entre `lda` et `pca_lda`.

In [ ]:
file_name_lda = 'knn_pca_lda_grid_search.pkl' if 'pca_lda' in implementation else 'knn_lda_grid_search.pkl'  # type: ignore
pipeline      = pipeline_pca_lda               if 'pca_lda' in implementation else pipeline_lda              # type: ignore

if not os.path.exists(f'{hyperparameter_dir}/{file_name_lda}'):
    prints.bold("Process : Cross Validation - LDA + KNN")
    grid_lda = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid_lda,
        cv=2,
        scoring=metrics.get_scoring(),
        refit='recall',
        n_jobs=1,
        verbose=2
    )
    grid_lda.fit(X_train, y_train)  # type: ignore
    joblib.dump(grid_lda, f'{hyperparameter_dir}/{file_name_lda}')

grid_lda = joblib.load(f'{hyperparameter_dir}/{file_name_lda}')

## Transformation LDA

On applique la transformation LDA du meilleur modèle sur les données d'entraînement et de test.

In [ ]:
X_train_lda = grid_lda.best_estimator_.named_steps['lda'].transform(X_train)  # type: ignore
X_test_lda  = grid_lda.best_estimator_.named_steps['lda'].transform(X_test)   # type: ignore

## Prédictions — LDA + KNN

On génère les prédictions sur les données de test avec le meilleur pipeline LDA + KNN.

In [ ]:
knn_lda    = grid_lda.best_estimator_
y_pred_lda = knn_lda.predict(X_test)  # type: ignore

## Résultats — LDA + KNN

On affiche et sauvegarde les résultats de la validation croisée pour le pipeline LDA.

In [ ]:
cross_validation_png(grid_lda, "LDA")
cross_validation_json(grid_lda, "LDA")